# Credit Card Fraud Detection Project

This notebook provides a comprehensive analysis and machine learning approach to detect credit card fraud. We will explore the dataset, perform exploratory data analysis (EDA), clean and engineer features, and train models to classify fraudulent transactions.

## Table of Contents
1. Setting Up the Virtual Environment
2. Importing Libraries
3. Loading the Dataset
4. Understanding the Dataset
5. Exploratory Data Analysis (EDA)
6. Data Visualization
7. Data Cleaning
8. Feature Engineering
9. Splitting the Dataset
10. Training Machine Learning Models
11. Evaluating Model Performance
12. Explaining Model Results

## 1. Setting Up the Virtual Environment

Before starting, we create a virtual environment to isolate our dependencies. This ensures that the packages we install do not conflict with other projects.

Run the following commands in your terminal:

```bash
python -m venv venv
source venv/Scripts/activate  # On Windows with Git Bash
pip install pandas numpy matplotlib seaborn scikit-learn jupyter imbalanced-learn
```

This setup provides all necessary libraries for data manipulation, visualization, and machine learning.

## 2. Importing Libraries

We import the necessary libraries for data handling, visualization, and machine learning. Each library serves a specific purpose:

- `pandas` and `numpy`: For data manipulation and numerical operations.
- `matplotlib` and `seaborn`: For creating visualizations.
- `scikit-learn`: For machine learning algorithms and evaluation metrics.
- `imbalanced-learn`: For handling imbalanced datasets.

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
from imblearn.over_sampling import SMOTE
import warnings
warnings.filterwarnings('ignore')

# Set style for plots
sns.set_style('whitegrid')
plt.style.use('seaborn')

## 3. Loading the Dataset

We load the credit card fraud dataset from the CSV file. This dataset contains transaction details with a fraud label. We handle potential encoding issues and inspect the initial data.

In [ ]:
# Load the dataset
df = pd.read_csv('credit_card_fraud.csv', encoding='utf-8')

# Display the first few rows
print("First 5 rows of the dataset:")
print(df.head())

# Check the shape
print(f"\nDataset shape: {df.shape}")

# Check data types
print(f"\nData types:\n{df.dtypes}")

## 4. Understanding the Dataset

We examine the dataset's structure, including the number of rows and columns, data types, and summary statistics. This helps us understand the features available for fraud detection.

In [ ]:
# Summary statistics
print("Summary statistics:")
print(df.describe())

# Check for missing values
print(f"\nMissing values per column:\n{df.isnull().sum()}")

# Unique values in categorical columns
categorical_cols = ['Merchant Category Code (MCC)', 'Transaction Currency', 'Card Type', 'Transaction Source', 'Device Information']
for col in categorical_cols:
    print(f"\nUnique values in {col}: {df[col].nunique()}")

# Fraud distribution
print(f"\nFraud distribution:\n{df['Fraud Flag or Label'].value_counts()}")

## 5. Exploratory Data Analysis (EDA)

EDA involves analyzing the data to find patterns, anomalies, and relationships. We check for correlations, distributions, and imbalances.

In [ ]:
# Correlation matrix for numerical features
numerical_cols = ['Transaction Amount']
corr = df[numerical_cols + ['Fraud Flag or Label']].corr()
print("Correlation matrix:")
print(corr)

# Check for outliers in transaction amount
plt.figure(figsize=(10, 6))
sns.boxplot(x='Fraud Flag or Label', y='Transaction Amount', data=df)
plt.title('Transaction Amount by Fraud Label')
plt.show()

# Distribution of transaction amounts
plt.figure(figsize=(10, 6))
sns.histplot(df['Transaction Amount'], bins=50, kde=True)
plt.title('Distribution of Transaction Amounts')
plt.show()

## 6. Data Visualization

Visualizations help in understanding the data better. We create plots for class distribution, correlations, and other insights.

In [ ]:
# Class distribution
plt.figure(figsize=(8, 6))
sns.countplot(x='Fraud Flag or Label', data=df)
plt.title('Fraud vs Non-Fraud Transactions')
plt.xlabel('Fraud Label (0: Non-Fraud, 1: Fraud)')
plt.ylabel('Count')
plt.show()

# Transaction amount by card type
plt.figure(figsize=(12, 6))
sns.boxplot(x='Card Type', y='Transaction Amount', data=df)
plt.title('Transaction Amount by Card Type')
plt.xticks(rotation=45)
plt.show()

# Fraud by transaction source
plt.figure(figsize=(10, 6))
sns.countplot(x='Transaction Source', hue='Fraud Flag or Label', data=df)
plt.title('Fraud Distribution by Transaction Source')
plt.show()

## 7. Data Cleaning

We clean the data by handling missing values, removing duplicates, and correcting inconsistencies. This ensures the dataset is ready for modeling.

In [ ]:
# Handle missing values
# For simplicity, drop rows with missing values in key columns
df_clean = df.dropna(subset=['Transaction Amount', 'Fraud Flag or Label'])

# Remove duplicates
df_clean = df_clean.drop_duplicates()

# Convert date to datetime
df_clean['Transaction Date and Time'] = pd.to_datetime(df_clean['Transaction Date and Time'])

# Extract useful features from date
df_clean['Hour'] = df_clean['Transaction Date and Time'].dt.hour
df_clean['Day'] = df_clean['Transaction Date and Time'].dt.day
df_clean['Month'] = df_clean['Transaction Date and Time'].dt.month

print(f"Shape after cleaning: {df_clean.shape}")

## 8. Feature Engineering

We create new features, encode categorical variables, and scale numerical features. We also handle class imbalance using SMOTE.

In [ ]:
# Select features
features = ['Transaction Amount', 'Merchant Category Code (MCC)', 'Card Type', 'Transaction Source', 'Hour', 'Day', 'Month']
target = 'Fraud Flag or Label'

# Encode categorical variables
le = LabelEncoder()
for col in ['Merchant Category Code (MCC)', 'Card Type', 'Transaction Source']:
    df_clean[col] = le.fit_transform(df_clean[col])

# Scale numerical features
scaler = StandardScaler()
df_clean[['Transaction Amount']] = scaler.fit_transform(df_clean[['Transaction Amount']])

# Prepare X and y
X = df_clean[features]
y = df_clean[target]

# Handle imbalance with SMOTE
smote = SMOTE(random_state=42)
X_res, y_res = smote.fit_resample(X, y)

print(f"Original class distribution: {y.value_counts()}")
print(f"Resampled class distribution: {pd.Series(y_res).value_counts()}")

## 9. Splitting the Dataset

We split the resampled data into training and testing sets to evaluate model performance.

In [ ]:
# Split the data
X_train, X_test, y_train, y_test = train_test_split(X_res, y_res, test_size=0.3, random_state=42, stratify=y_res)

print(f"Training set shape: {X_train.shape}")
print(f"Testing set shape: {X_test.shape}")

## 10. Training Machine Learning Models

We train Logistic Regression and Random Forest models, using grid search for hyperparameter tuning.

In [ ]:
# Logistic Regression
lr = LogisticRegression(random_state=42)
lr.fit(X_train, y_train)

# Random Forest
rf = RandomForestClassifier(random_state=42)
param_grid = {'n_estimators': [100, 200], 'max_depth': [10, 20]}
grid_rf = GridSearchCV(rf, param_grid, cv=3, scoring='roc_auc')
grid_rf.fit(X_train, y_train)

print(f"Best Random Forest params: {grid_rf.best_params_}")

## 11. Evaluating Model Performance

We evaluate the models using various metrics and visualize the results.

In [ ]:
# Predictions
lr_pred = lr.predict(X_test)
rf_pred = grid_rf.predict(X_test)

# Evaluation
print("Logistic Regression Report:")
print(classification_report(y_test, lr_pred))

print("Random Forest Report:")
print(classification_report(y_test, rf_pred))

# ROC AUC
lr_auc = roc_auc_score(y_test, lr.predict_proba(X_test)[:, 1])
rf_auc = roc_auc_score(y_test, grid_rf.predict_proba(X_test)[:, 1])

print(f"Logistic Regression AUC: {lr_auc:.4f}")
print(f"Random Forest AUC: {rf_auc:.4f}")

# Confusion Matrix
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
sns.heatmap(confusion_matrix(y_test, lr_pred), annot=True, fmt='d', ax=ax[0])
ax[0].set_title('Logistic Regression Confusion Matrix')
sns.heatmap(confusion_matrix(y_test, rf_pred), annot=True, fmt='d', ax=ax[1])
ax[1].set_title('Random Forest Confusion Matrix')
plt.show()

## 12. Explaining Model Results

The Random Forest model outperforms Logistic Regression in detecting fraud, with higher precision and recall. Feature importance shows transaction amount and time features are key. This model adds value by reducing financial losses from fraud.